In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scipy.io as sio
from dataclasses import dataclass
from typing import List, Tuple
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from natsort import natsorted
import numpy as np
import matplotlib.animation as animation
import xarray as xr
import h5py
import imageio
import matplotlib
import gc
import sys
import io
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from scipy.optimize import curve_fit
import scipy.integrate
import re
import scipy.ndimage

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(rf'../../../../tidy3d'))

from AutomationModule import * 

import AutomationModule as AM
plt.rcParams.update({'font.size': 22})  

tidy3dAPI = os.environ["API_TIDY3D_KEY"]
plt.rc('font', family='Arial')


In [ ]:
folder_path = r"../../../data/20260420 LSU Transmission n_3.4 ff_0.2172 ffh_0.225 Schulz"
data_file = rf"./data/20260420_ff_2172_elliptical_rods_n_3p4_schulz.h5"
data = AM.read_hdf5_as_dict(data_file)

In [ ]:
n_values = list(data['n_values'] )if 'n_values' in data.keys() else []
ff_values =list( data['ff']) if 'ff' in data.keys() else []
size_values = list(data['sizes'])if 'sizes' in data.keys() else []
z_values = list(data['z_values']) if 'z_values' in data.keys() else []
values = data['transmission_data'] if 'transmission_data' in data.keys() else {}
reference_entry=None
# Loop through all files in the folder
for dirpath, dirnames, filenames in os.walk(folder_path):
      try:
        z_value = float(re.search(r'z_([+-]?\d+(?:\.\d+)?)', dirpath).group(1))
      except AttributeError:
        print(f"Could not extract z_value from directory: {dirpath}")
        continue
      z_values.append(z_value)
      for filename in filenames:
        if reference_entry is None:
            print(filename)
            reference_object = AM.loadFromFile(key = tidy3dAPI, file_path=os.path.join(folder_path, "reference.txt"),get_ref=False)
            reference_entry = reference_object.sim_data['flux2'].flux
            reference_exit = reference_object.sim_data['flux1'].flux
        try:
            n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff = float(re.search(r'ffr_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            size = float(re.search(r'size_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            sample = float(re.search(r'sample_([+-]?\d+(?:\.\d+)?)', filename).group(1))
            ff_values.append(ff)
            n_values.append(n_value)
            size_values.append(size)
            #Retrieve simulation data 
            if os.path.isfile(os.path.join(dirpath, filename)):
              file=os.path.join(dirpath, filename)
              structure_1 = AM.loadFromFile(key = tidy3dAPI, file_path=file,get_ref=False)
              sim_data_i = structure_1.sim_data
              transmission_entry = sim_data_i['flux2'].flux
              transmission_exit = sim_data_i['flux1'].flux
              if str(n_value) not in values.keys():
                  values[str(n_value)] = {}
              if str(ff) not in values[str(n_value)].keys():
                  values[str(n_value)][str(ff)] = {}
              if str(z_value) not in values[str(n_value)][str(ff)].keys():
                    values[str(n_value)][str(ff)][str(z_value)] = {}
              if str(size) not in values[str(n_value)][str(ff)][str(z_value)].keys():
                    values[str(n_value)][str(ff)][str(z_value)][str(size)] = {}
              if str(sample) not in values[str(n_value)][str(ff)][str(z_value)][str(size)].keys():
                    values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)] = {}

              values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)]['entry'] = transmission_entry.values/reference_entry.values
              values[str(n_value)][str(ff)][str(z_value)][str(size)][str(sample)]['exit'] = transmission_exit.values/reference_exit.values
        except Exception as e:
            print("Error:", e)
            continue
       

# After the loop, get unique values as arrays
n_unique = np.unique(n_values)
ff_unique = np.unique(ff_values)
size_unique = np.unique(size_values)
z_unique = np.unique(z_values)




In [ ]:
print("Unique n values:", n_unique)
print("Unique z values:", z_unique)
print("Unique ff values:", ff_unique)
print("Unique size values:", np.asanyarray(size_unique)*11.44)

In [ ]:
data = {
        'transmission_data':values,
        "raw_freqs":sim_data_i.simulation.monitors[0].freqs,
        "sizes":np.array(size_unique),
        "ff":np.array(ff_unique),
        "cell_size":11.44,
        "z_values":np.array(z_unique),
        "n_values":np.array(n_unique)
    }



In [ ]:
data["transmission_data"]['3.4']['0.2172']['100.0'].keys()

In [ ]:
os.makedirs("./data",exist_ok=True)

create_hdf5_from_dict(data,data_file)